# Посадка на TRAPPIST-1d


Заготовка: заполните пропуски. Строки `print` менять не нужно — они печатают
ответ ровно в том виде, в каком его надо вписать в форму.

**Система координат.** Везде используется индексация NumPy с нуля: верхняя левая клетка — `(0, 0)`. Координаты записываются в порядке `(строка, столбец)`. Кадры тоже нумеруются с нуля: `frames[0]` — кадр № 0.

## С чего начать

In [ ]:
import numpy as np
import hashlib

frames  = np.load("frames.npy")      # (21, 80, 120) — стек снимков
offsets = np.load("offsets.npy")     # (21,) — сдвиг сканера на каждом пролёте
station = tuple(np.load("station.npy"))
patch   = np.load("patch.npy")       # (8, 8) — кусочек карты станции
routes  = np.load("routes.npz")

print("весь стек:      ", frames.shape)
print("один кадр:      ", frames[0].shape)
print("история пикселя:", frames[:, 40, 60].shape)

# три контрольные суммы — из стартового блока задания
CHECKS = {
    "кадры":    "0f68c70514fdd0da370c0717ef7049468ebaebab67fb4f441e754e6662f5bf78",
    "площадка": "27a927537cc199c0090663e9e7f35e9225514d1cf93b2d41d31471c1ab8d58f4",
    "маршрут":  "7ef4ad11ffece5e5a9608faa27495ac0094fe09e3b6c1ac24e2db3a39ea5ef89",
}

весь стек:       (21, 80, 120)
один кадр:       (80, 120)
история пикселя: (21,)


## 1. Что сломалось

**(1а)** сколько застывших пикселей — целое число.
**(1б)** номера бракованных кадров с нуля через запятую по возрастанию, без пробелов.

In [ ]:
# застывший пиксель: во всех 21 кадре одно и то же конечное число
stuck= (frames == frames[0]).all(axis=0)

         # булева маска формы (80, 120)

print("(1а)", int(stuck.sum()))

(1а) 43


In [ ]:
# зашкал: frames >= 1000; кадр бракуем, если зашкалило больше 20% клеток

flare_share = (frames >= 1000).mean(axis=(1, 2))


bad_frames = np.where(flare_share > 0.20)[0]

print("(1б)", ",".join(map(str, bad_frames)))

(1б) 3,11,17


## 2. Чистая карта

**(2а)** сколько клеток карты остались неизвестными — целое число.
**(2б)** наибольшая известная высота в метрах, с одним знаком после точки.

In [ ]:
clean = frames.astype(float).copy()

clean[:, stuck] = np.nan
clean[bad_frames, :, :] = np.nan
clean[clean >= 1000] = np.nan

clean = clean - offsets[:, None, None]

valid_count = np.isfinite(clean).sum(axis=0)

height = np.nanmedian(clean, axis=0)

height[valid_count < 5] = np.nan

print("(2a)", int(np.isnan(height).sum()))
print("(2б)", round(float(np.nanmax(height)), 1))

(2a) 67
(2б) 127.4


/tmp/ipykernel_1575/2373527837.py:11: RuntimeWarning: All-NaN slice encountered
  height = np.nanmedian(clean, axis=0)


## 3. Уклоны

**(3а)** наибольший уклон в метрах, с одним знаком после точки.

In [ ]:
slope = np.full_like(height, np.nan)

center = height[1:-1, 1:-1]

d_top    = np.abs(center - height[:-2, 1:-1])
d_bottom = np.abs(center - height[2:, 1:-1])
d_left   = np.abs(center - height[1:-1, :-2])
d_right  = np.abs(center - height[1:-1, 2:])

slope[1:-1, 1:-1] = np.max(np.stack([d_top, d_bottom, d_left, d_right]), axis=0)


print("(3a)", round(float(np.nanmax(slope)), 1))

(3a) 30.9


## 4. Место для корабля

**(4а)** сколько площадок подходит — целое число.
**(4б)** центр выбранной: строка и столбец (оба с нуля) через запятую.

In [ ]:
safe = (height < 100) & (slope <= 2)

ok = (
    safe[0:-2, 0:-2] & safe[0:-2, 1:-1] & safe[0:-2, 2:] &
    safe[1:-1, 0:-2] & safe[1:-1, 1:-1] & safe[1:-1, 2:] &
    safe[2:  , 0:-2] & safe[2:  , 1:-1] & safe[2:  , 2:]
)

h_slices = [height[i:i+78, j:j+118] for i in range(3) for j in range(3)]
h_stack = np.stack(h_slices, axis=0)  # форма (9, 78, 118)
spread = np.max(h_stack, axis=0) - np.min(h_stack, axis=0)

valid_spread = np.where(ok, spread, np.inf)

min_flat_idx = np.argmin(valid_spread)
r_top, c_left = np.unravel_index(min_flat_idx, valid_spread.shape)

pad = (r_top + 1, c_left + 1)

print("(4a)", int(ok.sum()))
print("(4б)", f"{pad[0]},{pad[1]}")

(4a) 261
(4б) 47,72


## 5. Дорога к станции ★

*Необязательная задача.*

**(5а)** сколько маршрутов допустимо — целое число.
**(5б)** ключ выбранного маршрута в архиве, например `route_02`.
**(5в)** его длина в метрах — целое число.

In [ ]:
def path_length(P):
    """Длина пути в метрах или None, если путь не годится."""
    # границы карты -> старт и финиш -> шаги по стороне -> все клетки безопасны
    ...

good = {}            # ключ маршрута -> длина, только допустимые пути
for name in routes.files:
    ...

best = ...           # ключ кратчайшего допустимого

print("(5а)", len(good))
print("(5б)", best)
print("(5в)", good[best])

## 6. Радиограмма на станцию ★

*Необязательная задача.*

**(6а)** координаты площадки в системе станции: строка и столбец (оба с нуля) через запятую.

In [ ]:
# восемь вариантов: np.rot90 на 0..3 поворота, каждый с np.fliplr и без
# берём тот, где верхний угол 8x8 ближе всего к patch
turned = ...         # ваша карта в системе станции

# правило перевода координат — по трём точкам: (0,0), (1,0), (0,1)
r2, c2 = ..., ...    # центр площадки в системе станции (считая с нуля)

print("(6а)", f"{r2},{c2}")

## Самопроверка

In [ ]:
def h(value):
    return hashlib.sha256(value.encode()).hexdigest()

mine = {
    "кадры": ",".join(map(str, bad_frames)),
    "площадка": f"{pad[0]},{pad[1]}",
}
if "best" in globals():
    mine["маршрут"] = best

for name, value in mine.items():
    print(f"{name:10}", "сошлось" if h(value) == CHECKS[name] else "НЕ СОШЛОСЬ")

кадры      сошлось
площадка   сошлось
